# Tutorial: SSL Detection Lab on Football Players with 2× T4 GPUs

**Audience:** computer-vision students who know basic Python and object detection but are new to self-supervised learning (SSL), distributed training, or systematic evaluation.

**Dataset:** `iasadpanwhar/football-player-detection-yolov8` on Kaggle.

**Prerequisites**

- Create a Kaggle notebook, click **Add Input**, choose **Datasets** (not Competitions), and attach the dataset named above.
- In **Settings → Accelerator**, select **GPU T4 ×2**.
- Enable Internet if the library or model files are not attached as a Kaggle dataset.

**Learning goals**

By the end you can:

1. audit a YOLO-format dataset and create a correct dataset YAML;
2. list the library's model and SSL capabilities;
3. launch two-GPU SSL pretraining with `torchrun`;
4. dry-run every SSL method and every supported model family without claiming research results;
5. fine-tune one SSL backbone, evaluate it, and analyse a short video.

> **Dry-run boundary:** every training job uses one epoch and a tiny image subset. The purpose is to verify code, shapes, checkpoints, and GPU distribution. The resulting accuracy is not scientific evidence.

## Tutorial outline

1. Install and verify the environment
2. Audit the football dataset
3. Display supported SSL methods and model families
4. Configure reproducible dry runs
5. Run every SSL method on one reference backbone
6. Run every SSL-compatible model family with one reference SSL method
7. Smoke-test inference-only families
8. Fine-tune, evaluate, and run video analysis
9. Review pitfalls and complete exercises

## 1. Install and verify the current library

The setup cell uses an attached source folder when one exists; otherwise it installs the current
GitHub `main` branch. Students should see **ssl-detection-lab 0.4.0 or newer**. Version 0.4 uses
`torch.amp`, secure checkpoint loading, and torchvision transforms v2.

Kaggle already supplies a CUDA-matched PyTorch build. Do not independently replace it with a
generic PyTorch wheel. The package installer only upgrades dependencies when the runtime does not
satisfy the supported minimum.


In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path

# Prefer source attached as a Kaggle Dataset; fall back to the public repository.
LOCAL_SOURCES = [
    Path("/kaggle/working/ssl-detection-lab"),
    Path("/kaggle/input/ssl-detection-lab"),
    Path("/kaggle/input/ssl-detection-lab/ssl-detection-lab"),
]
GITHUB_SOURCE = "git+https://github.com/rifat963/ssl-detection-lab.git@main"
LIBRARY_SOURCE = next(
    (str(path) for path in LOCAL_SOURCES if (path / "pyproject.toml").is_file()),
    GITHUB_SOURCE,
)

print("Installing ssl-detection-lab from:", LIBRARY_SOURCE)
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "--no-cache-dir",
    LIBRARY_SOURCE,
])

# Clear a previously imported copy if this cell is re-run in the same kernel.
for module_name in list(sys.modules):
    if module_name == "ssldet" or module_name.startswith("ssldet."):
        del sys.modules[module_name]
importlib.invalidate_caches()

import ssldet

version = tuple(int(part) for part in ssldet.__version__.split(".")[:3])
if version < (0, 8, 1) or not hasattr(ssldet, "capabilities"):
    raise RuntimeError(
        "The installed library is too old. Attach the updated source folder or install "
        "ssl-detection-lab v0.8.1+ from GitHub."
    )

print("ssl-detection-lab version:", ssldet.__version__)
print("Loaded from:", ssldet.__file__)
print("Public API verified: capabilities() is available")


### Verify the current runtime and both T4 GPUs

`assert_supported_runtime()` is the library's environment doctor. It reports exact dependency,
CUDA, cuDNN, and GPU information, then stops early if this is an old or CPU-only runtime.


In [ ]:
import json
import random

import numpy as np
import torch
from ssldet import assert_supported_runtime

runtime = assert_supported_runtime(require_cuda=True, minimum_gpus=2)
print(json.dumps(runtime, indent=2))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Audit the football-player dataset

SSL pretraining must read **only `train/images`**. Validation and test images remain untouched until downstream evaluation, preventing data leakage.

In [ ]:
from collections import Counter

DATASET_ROOT_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/"
        "football_players_detection/football_players_detection"
    ),
    Path(
        "/kaggle/input/football-player-detection-yolov8/"
        "football_players_detection/football_players_detection"
    ),
]

def is_dataset_root(path: Path) -> bool:
    return all(
        (path / relative).is_dir()
        for relative in ("train/images", "train/labels", "valid/images", "test/images")
    )


DATASET_ROOT = next(
    (path for path in DATASET_ROOT_CANDIDATES if is_dataset_root(path)),
    None,
)
if DATASET_ROOT is None:
    discovered = [
        train_images.parent.parent
        for train_images in Path("/kaggle/input").glob("**/train/images")
        if is_dataset_root(train_images.parent.parent)
    ]
    DATASET_ROOT = discovered[0] if discovered else None
if DATASET_ROOT is None:
    mounted = [str(path) for path in Path("/kaggle/input").iterdir()]
    raise FileNotFoundError(
        "Dataset is not mounted. In Kaggle click Add Input -> Datasets, add "
        "iasadpanwhar/football-player-detection-yolov8, then rerun. "
        f"Mounted inputs: {mounted}"
    )
print("Using dataset root:", DATASET_ROOT)
SPLITS = {
    "train": {
        "images": DATASET_ROOT / "train/images",
        "labels": DATASET_ROOT / "train/labels",
    },
    "val": {
        "images": DATASET_ROOT / "valid/images",
        "labels": DATASET_ROOT / "valid/labels",
    },
    "test": {
        "images": DATASET_ROOT / "test/images",
        "labels": DATASET_ROOT / "test/labels",
    },
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

audit_rows = []
for split, paths in SPLITS.items():
    assert paths["images"].is_dir(), f"Missing directory: {paths['images']}"
    assert paths["labels"].is_dir(), f"Missing directory: {paths['labels']}"
    images = [p for p in paths["images"].iterdir() if p.suffix.lower() in IMAGE_SUFFIXES]
    labels = list(paths["labels"].glob("*.txt"))
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    audit_rows.append(
        {
            "split": split,
            "images": len(images),
            "labels": len(labels),
            "images_without_labels": len(image_stems - label_stems),
            "labels_without_images": len(label_stems - image_stems),
        }
    )

import pandas as pd

audit = pd.DataFrame(audit_rows)
display(audit)

### Recover class names and write a Kaggle-working YAML

The code first looks for a dataset YAML. If none exists, it inspects class IDs in label files. The common four-class football mapping is used only when exactly four IDs are found; otherwise neutral `class_0`, `class_1`, … names are used so students must verify them.

In [ ]:
import yaml

def read_class_ids(label_dir: Path) -> list[int]:
    ids = set()
    for label_path in label_dir.glob("*.txt"):
        for row in label_path.read_text(encoding="utf-8").splitlines():
            fields = row.split()
            if fields:
                ids.add(int(float(fields[0])))
    return sorted(ids)


yaml_candidates = list(DATASET_ROOT.glob("*.yaml")) + list(DATASET_ROOT.glob("*.yml"))
source_yaml = yaml_candidates[0] if yaml_candidates else None
names = None
if source_yaml:
    source_config = yaml.safe_load(source_yaml.read_text(encoding="utf-8")) or {}
    raw_names = source_config.get("names")
    if isinstance(raw_names, dict):
        names = [str(raw_names[index]) for index in sorted(raw_names)]
    elif isinstance(raw_names, list):
        names = [str(name) for name in raw_names]

class_ids = read_class_ids(SPLITS["train"]["labels"])
if names is None:
    if class_ids == [0, 1, 2, 3]:
        names = ["ball", "goalkeeper", "player", "referee"]
    else:
        names = [f"class_{index}" for index in range(max(class_ids) + 1)]

assert class_ids == list(range(len(names))), (
    f"Label IDs {class_ids} do not match class names {names}. Correct names before training."
)

DATA_YAML = Path("/kaggle/working/football_players_detection.yaml")
data_config = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {index: name for index, name in enumerate(names)},
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

### Visual label sanity check

YOLO labels contain normalized `class x_center y_center width height`. Draw several boxes before spending GPU time; incorrect class order or box scaling can invalidate every later metric.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

sample_image = sorted(
    p for p in SPLITS["train"]["images"].iterdir() if p.suffix.lower() in IMAGE_SUFFIXES
)[0]
sample_label = SPLITS["train"]["labels"] / f"{sample_image.stem}.txt"

image = Image.open(sample_image).convert("RGB")
figure, axis = plt.subplots(figsize=(12, 7))
axis.imshow(image)
width, height = image.size
if sample_label.exists():
    for row in sample_label.read_text(encoding="utf-8").splitlines():
        class_id, x_center, y_center, box_width, box_height = map(float, row.split()[:5])
        x = (x_center - box_width / 2) * width
        y = (y_center - box_height / 2) * height
        rectangle = patches.Rectangle(
            (x, y), box_width * width, box_height * height,
            linewidth=2, edgecolor="lime", facecolor="none",
        )
        axis.add_patch(rectangle)
        axis.text(x, y, names[int(class_id)], color="black", backgroundcolor="lime")
axis.set_title(sample_image.name)
axis.axis("off")
plt.show()

## 3. Start with the library capability catalog

`capabilities()` is the first call students should make. It separates:

- **SSL-compatible YOLO backbones**, which can be pretrained without labels;
- **evaluation/video models**, which can run downstream inference;
- **official DINOv2 ViT feature backbones**, which return embeddings rather than boxes.

The main public calls used in this tutorial are `make_dry_run_config`,
`launch_distributed_pretrain`, `evaluate`, and `analyze_video`.


In [ ]:
import pandas as pd
import ssldet
from ssldet import capabilities

catalog = capabilities()
print("ssl-detection-lab version:", ssldet.__version__)

print("\nSupported SSL architectures")
display(pd.DataFrame(catalog["ssl_architectures"]))

print("\nSupported detector families")
display(
    pd.DataFrame(catalog["model_families"])[
        ["name", "scales", "ssl_backbone", "dataset_evaluation", "video_analysis", "tasks"]
    ]
)

print("\nOfficial DINOv2 feature backbones")
display(pd.DataFrame(catalog["dinov2_feature_backbones"]))


### What each SSL method teaches

| Method | Main idea | Useful first knobs |
|---|---|---|
| SimCLR | Match two augmented views using contrastive negatives | `temperature`, batch size |
| BYOL | Predict an EMA target representation without negatives | `momentum` |
| MoCo | Contrast against a momentum queue | `temperature`, `queue_size` |
| DINOv2-style | Multi-crop centered teacher–student distillation + KoLeo | crop count, temperatures |
| MAE | Reconstruct masked pixels from spatial features | `mask_ratio` |
| I-JEPA | Predict masked target regions in latent space | target blocks, predictor depth |

The DINOv2 implementation is a documented, compute-scaled YOLO adaptation. It is not a reproduction of official ViT+iBOT training on LVD-142M.

## 4. Configure safe dry runs

The two matrices below cover every method and family without running every method on every family. Set either flag to `False` when demonstrating only selected cells in class.

In [ ]:
from tqdm.auto import tqdm

RUN_SSL_METHOD_DRY_RUNS = True
RUN_MODEL_FAMILY_DRY_RUNS = True
RUN_INFERENCE_ONLY_SMOKES = True
RUN_DOWNSTREAM_DEMO = True

DRY_EPOCHS = 1
DRY_MAX_IMAGES = 32
DRY_IMAGE_SIZE = 128
DRY_BATCH_PER_GPU = 4
DRY_WORKERS = 2

WORK_ROOT = Path("/kaggle/working/ssldet_football_tutorial")
CONFIG_ROOT = WORK_ROOT / "configs"
RUN_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("Dry-run outputs:", RUN_ROOT)

### Discover model YAMLs available in the installed Ultralytics version

Model availability changes between Ultralytics releases. Each family has ordered candidates; the first architecture that can be constructed is selected. Unsupported candidates are reported and skipped instead of stopping the tutorial.

In [ ]:
from ultralytics import YOLO

MODEL_CANDIDATES = {
    "YOLO26": ["yolo26n.yaml"],
    "YOLO12": ["yolo12n.yaml"],
    "YOLO11": ["yolo11n.yaml"],
    "YOLOv10": ["yolov10n.yaml"],
    "YOLOv9": ["yolov9t.yaml", "yolov9c.yaml"],
    "YOLOv8": ["yolov8n.yaml"],
    "YOLOv6": ["yolov6n.yaml"],
    "YOLOv5u": ["yolov5nu.yaml", "yolov5n.yaml"],
    "YOLOv3u": ["yolov3u.yaml", "yolov3.yaml"],
}

def first_available_yaml(candidates: list[str]) -> tuple[str | None, str]:
    errors = []
    for candidate in candidates:
        try:
            YOLO(candidate)
            return candidate, "available"
        except Exception as error:
            errors.append(f"{candidate}: {type(error).__name__}: {error}")
    return None, " | ".join(errors)


model_probe_rows = []
available_models = {}
for family, candidates in tqdm(
    MODEL_CANDIDATES.items(),
    total=len(MODEL_CANDIDATES),
    desc="Model YAML probe",
    unit="family",
):
    selected, status = first_available_yaml(candidates)
    if selected:
        available_models[family] = selected
    model_probe_rows.append(
        {"family": family, "selected_yaml": selected, "status": status}
    )

model_probe = pd.DataFrame(model_probe_rows)
display(model_probe)
assert available_models, "No compatible YOLO YAML was found. Update ultralytics."

REFERENCE_FAMILY = next(
    (
        family
        for family in ("YOLO26", "YOLO11", "YOLOv8")
        if family in available_models
    ),
    next(iter(available_models)),
)
REFERENCE_MODEL = available_models[REFERENCE_FAMILY]
print("Reference backbone:", REFERENCE_FAMILY, REFERENCE_MODEL)

### Create and launch a dry run with two library calls

`make_dry_run_config(...)` creates a validated one-epoch, low-memory configuration.
`launch_distributed_pretrain(..., num_processes=2)` saves its YAML and launches one process on
each T4. The helper below only converts the result into a table row for this tutorial.

- A `.yaml` model starts randomly and is strict label-free SSL.
- A `.pt` model is a supervised warm start for domain adaptation.


In [ ]:
import json

from ssldet import launch_distributed_pretrain, make_dry_run_config


def run_ddp_dry_run(method: str, model_yaml: str, run_name: str) -> dict:
    """Run one tiny SSL job and return one student-friendly result row."""
    output_dir = RUN_ROOT / run_name
    config = make_dry_run_config(
        method=method,
        train_images=SPLITS["train"]["images"],
        output_dir=output_dir,
        yolo_model=model_yaml,
        max_images=DRY_MAX_IMAGES,
        image_size=DRY_IMAGE_SIZE,
        batch_size=DRY_BATCH_PER_GPU,
        workers=DRY_WORKERS,
        seed=SEED,
    )
    result = launch_distributed_pretrain(
        config,
        num_processes=2,
        config_path=CONFIG_ROOT / f"{run_name}.yaml",
        check=False,
    )

    manifest_path = output_dir / "run_manifest.json"
    manifest = (
        json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest_path.exists()
        else {}
    )
    return {
        "run": run_name,
        "method": method,
        "model": model_yaml,
        "status": "passed" if result.succeeded else "failed",
        "return_code": result.return_code,
        "seconds": round(result.seconds, 1),
        "best_loss": manifest.get("best_loss"),
        "output_dir": str(result.output_dir),
    }


## 5. Dry-run every SSL method

Every self-contained method uses the same reference backbone and the same 32 training images. DINOv3-guided distillation is covered in its dedicated notebook because it requires an authorized teacher checkpoint. Comparing these losses numerically is not meaningful because each objective has a different scale; this cell only checks that every pipeline completes and saves a checkpoint.

In [ ]:
SSL_METHODS = [
    item["name"] for item in catalog["ssl_architectures"]
    if item["name"] != "dinov3"
]
method_results = []

if RUN_SSL_METHOD_DRY_RUNS:
    for method in tqdm(SSL_METHODS, desc="SSL method dry runs", unit="method"):
        print(f"\n{'=' * 80}\nDry-running {method.upper()} on {REFERENCE_MODEL}\n")
        method_results.append(
            run_ddp_dry_run(
                method,
                REFERENCE_MODEL,
                f"methods/{method}_{Path(REFERENCE_MODEL).stem}",
            )
        )
else:
    print("Method dry runs disabled. Set RUN_SSL_METHOD_DRY_RUNS=True to execute them.")

method_results_df = pd.DataFrame(method_results)
display(method_results_df)

**Interpretation:** `passed` means the two-GPU data loader, augmentations, loss, optimizer, checkpoint, and YOLO-backbone transfer completed. One epoch is not enough to judge representation quality.

## 6. Dry-run every SSL-compatible model family

SimCLR is used as the common lightweight method. The reference family is reused from the previous matrix to avoid repeating exactly the same job. A missing YAML is marked unavailable by the probe rather than misrepresented as a library failure.

In [ ]:
family_results = []

if RUN_MODEL_FAMILY_DRY_RUNS:
    for family, model_yaml in tqdm(
    available_models.items(),
    total=len(available_models),
    desc="Model-family dry runs",
    unit="family",
):
        if family == REFERENCE_FAMILY and RUN_SSL_METHOD_DRY_RUNS:
            reused = next(
                (row for row in method_results if row["method"] == "simclr"),
                None,
            )
            if reused:
                family_results.append({"family": family, **reused, "status": "reused"})
                continue
        print(f"\n{'=' * 80}\nDry-running {family}: {model_yaml}\n")
        result = run_ddp_dry_run(
            "simclr",
            model_yaml,
            f"models/{family.lower()}_simclr_{Path(model_yaml).stem}",
        )
        family_results.append({"family": family, **result})
else:
    print("Family dry runs disabled. Set RUN_MODEL_FAMILY_DRY_RUNS=True to execute them.")

family_results_df = pd.DataFrame(family_results)
display(family_results_df)

### Custom YOLO models

“Custom Ultralytics YOLO” cannot be dry-run without a supplied architecture. To test one, add its YAML path to `MODEL_CANDIDATES`, or call:

```python
run_ddp_dry_run("dinov2", "/kaggle/input/my-model/custom.yaml", "custom/dinov2")
```

Compatibility requires an Ultralytics YAML backbone whose final backbone feature is spatial (`B×C×H×W`).

## 7. Smoke-test evaluation/video-only families

RT-DETR and YOLO-NAS are not connected to the YOLO SSL backbone adapter. The following tests only load official weights and predict one dataset image. Downloads require Internet. Failures are recorded without stopping the tutorial because some Kaggle images omit optional YOLO-NAS dependencies.

In [ ]:
inference_only_rows = []
if RUN_INFERENCE_ONLY_SMOKES:
    inference_models = [
        ("RT-DETR", "rtdetr-l.pt", "RTDETR"),
        ("YOLO-NAS", "yolo_nas_s.pt", "NAS"),
    ]
    for family, weights, loader_name in tqdm(
    inference_models, desc="Inference-only smoke tests", unit="model"
):
        try:
            if loader_name == "RTDETR":
                from ultralytics import RTDETR
                model = RTDETR(weights)
            else:
                from ultralytics import NAS
                model = NAS(weights)
            prediction = model.predict(
                source=str(sample_image), imgsz=320, device=0, verbose=False
            )[0]
            boxes = prediction.boxes
            inference_only_rows.append(
                {
                    "family": family,
                    "weights": weights,
                    "status": "passed",
                    "detections": 0 if boxes is None else len(boxes),
                }
            )
        except Exception as error:
            inference_only_rows.append(
                {
                    "family": family,
                    "weights": weights,
                    "status": "skipped/failed",
                    "reason": f"{type(error).__name__}: {error}",
                }
            )

display(pd.DataFrame(inference_only_rows))

### Official DINOv2 feature-backbone smoke test

This is feature extraction—not detection. ViT-S/14 is used to keep the download and memory footprint small. A detector head must be attached before video boxes or mAP can be produced.

In [ ]:
if RUN_INFERENCE_ONLY_SMOKES:
    try:
        from ssldet import build_dinov2_transform, load_dinov2_backbone

        preprocess = build_dinov2_transform(224)
        image_tensor = preprocess(Image.open(sample_image).convert("RGB")).unsqueeze(0).cuda()
        dino_encoder = load_dinov2_backbone(
            "dinov2_vits14", device="cuda", freeze=True
        )
        with torch.inference_mode():
            embedding = dino_encoder(image_tensor)
            feature_map = dino_encoder.forward_feature_map(image_tensor)
        print("Global embedding:", tuple(embedding.shape))
        print("Dense feature map:", tuple(feature_map.shape))
        del dino_encoder, image_tensor, embedding, feature_map
        torch.cuda.empty_cache()
    except Exception as error:
        print("DINOv2 smoke test skipped/failed:", type(error).__name__, error)

## 8. Downstream detector dry run

SSL pretraining learns a backbone, not a finished football detector. This section takes the DINOv2-style student backbone, fine-tunes the detector for one tiny supervised epoch, evaluates labelled test data, and analyses a short video made from test frames.

For a real experiment, increase epochs and labelled fraction, keep settings identical across SSL methods, and repeat at least three seeds.

In [ ]:
from ultralytics import YOLO

DOWNSTREAM_ROOT = WORK_ROOT / "downstream"
SSL_DINO_DIR = RUN_ROOT / f"methods/dinov2_{Path(REFERENCE_MODEL).stem}"
SSL_DINO_WEIGHTS = SSL_DINO_DIR / f"dinov2_pretrained_{Path(REFERENCE_MODEL).stem}.pt"
BEST_DETECTOR = None

if RUN_DOWNSTREAM_DEMO and SSL_DINO_WEIGHTS.exists():
    detector = YOLO(str(SSL_DINO_WEIGHTS))
    detector.train(
        data=str(DATA_YAML),
        epochs=1,
        fraction=0.05,
        imgsz=320,
        batch=8,
        device=[0, 1],
        workers=2,
        amp=True,
        plots=False,
        project=str(DOWNSTREAM_ROOT),
        name="dinov2_detector_dry_run",
        exist_ok=True,
        seed=SEED,
    )
    BEST_DETECTOR = DOWNSTREAM_ROOT / "dinov2_detector_dry_run/weights/best.pt"
    print("Detector checkpoint:", BEST_DETECTOR)
else:
    print(
        "Downstream demo skipped. Run the DINOv2 method dry run first, or set "
        "RUN_DOWNSTREAM_DEMO=True."
    )

### Comprehensive labelled evaluation

Unlike an unlabelled video, this test split can produce precision, recall, F1, AP/mAP, per-class metrics, per-image metrics, curves, speed, and a confusion matrix.

In [ ]:
from ssldet import EvaluationConfig, evaluate

evaluation_result = None
if BEST_DETECTOR and BEST_DETECTOR.exists():
    evaluation_result = evaluate(
        EvaluationConfig(
            model_name="custom",
            weights_file=str(BEST_DETECTOR),
            data=str(DATA_YAML),
            output_dir=str(DOWNSTREAM_ROOT / "evaluation"),
            split="test",
            image_size=320,
            batch_size=8,
            device="0,1",
            workers=2,
            plots=True,
        )
    )
    report = json.loads(evaluation_result.metrics_json.read_text(encoding="utf-8"))
    display(pd.DataFrame([report["headline_metrics"]]))
    display(pd.DataFrame(report["per_class"]))
    print("Full report:", evaluation_result.metrics_json)

### Build a short video from test images and analyse it

This synthetic preview is only for exercising the video API. Detection counts, confidence, box occupancy, latency, throughput, and track statistics are valid outputs. Tracking-accuracy metrics such as HOTA or IDF1 require ground-truth identities, which YOLO detection labels do not provide.

In [ ]:
import cv2

PREVIEW_VIDEO = WORK_ROOT / "football_test_preview.mp4"
test_images = sorted(
    p for p in SPLITS["test"]["images"].iterdir() if p.suffix.lower() in IMAGE_SUFFIXES
)[:24]

first_frame = cv2.imread(str(test_images[0]))
frame_height, frame_width = first_frame.shape[:2]
writer = cv2.VideoWriter(
    str(PREVIEW_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    6.0,
    (frame_width, frame_height),
)
for image_path in test_images:
    frame = cv2.imread(str(image_path))
    if frame.shape[:2] != (frame_height, frame_width):
        frame = cv2.resize(frame, (frame_width, frame_height))
    writer.write(frame)
writer.release()

print("Preview video:", PREVIEW_VIDEO, "frames:", len(test_images))

In [ ]:
from ssldet import VideoAnalysisConfig, analyze_video

video_result = None
if BEST_DETECTOR and BEST_DETECTOR.exists():
    video_result = analyze_video(
        VideoAnalysisConfig(
            video_source=str(PREVIEW_VIDEO),
            model_name="custom",
            weights_file=str(BEST_DETECTOR),
            output_dir=str(DOWNSTREAM_ROOT / "video_analysis"),
            confidence=0.15,
            image_size=320,
            device=0,
            tracker="botsort.yaml",
            max_frames=len(test_images),
            save_annotated=True,
        )
    )
    video_report = json.loads(video_result.report_json.read_text(encoding="utf-8"))
    display(pd.DataFrame([video_report["video_metrics"]]))
    display(pd.DataFrame(video_report["per_class"]))
    print("Readable outcome:", video_result.outcome_markdown)

## Common pitfalls

- **Old package import:** the first cell must print v0.8.1+ and verify `capabilities()`.
- **Kaggle permission error:** attach the input from **Datasets**, not Competitions; this notebook
  intentionally has no competition metadata.
- **AMP warning:** `torch.cuda.amp` means an old `ssldet` is still imported; restart the kernel and rerun the v0.8 installation cell.
- **Leakage:** never SSL-pretrain on `valid/images` or `test/images` for a standard evaluation.
- **Wrong initialization claim:** `.yaml` is random initialization; `.pt` is a supervised warm start.
- **Dry-run overinterpretation:** one epoch and 32 images validate plumbing, not model quality.
- **DINOv2 confusion:** DINOv2 is an SSL feature learner; DINO can also name a DETR detector.
- **Raw SSL checkpoint:** fine-tune the exported `*_pretrained_*.pt` detector, not `best_ssl.pt`.
- **Out of memory:** reduce per-GPU batch, local crops, image size, or projection dimensions.
- **Legacy checkpoint:** use current Ultralytics YOLOv5u/YOLOv3u files.


## Exercises

1. Change `DRY_MAX_IMAGES` from 32 to 128 and predict which methods will slow down most.
2. Run BYOL with `momentum=0.99` and `0.999`; compare only within BYOL.
3. Fine-tune SimCLR and DINOv2-style backbones with identical detector settings and three seeds.
4. Add a custom YOLO architecture to `MODEL_CANDIDATES` and explain whether its final backbone output is spatial.
5. Explain why mAP cannot be computed from a video link without labels.

### Exercise answer scaffold

The next cell collects training histories without pretending losses from different objectives are directly comparable. Extend it with downstream mAP after running controlled fine-tuning experiments.

In [ ]:
history_rows = []
for history_path in RUN_ROOT.rglob("history.csv"):
    history = pd.read_csv(history_path)
    if not history.empty:
        history_rows.append(
            {
                "run": str(history_path.parent.relative_to(RUN_ROOT)),
                "epochs": len(history),
                "final_ssl_loss": history.iloc[-1]["loss"],
                "seconds": history["seconds"].sum(),
            }
        )

histories = pd.DataFrame(history_rows)
display(histories.sort_values("run") if not histories.empty else histories)

# Student task: join this table with controlled downstream mAP50-95 results.

## Next steps for a real study

- Increase SSL epochs and use all training images.
- Fine-tune every initialization with identical detector hyperparameters.
- Compare random, COCO-supervised, strict SSL, and COCO+SSL initialization.
- Repeat at least three seeds and report mean ± standard deviation.
- Report mAP50-95, mAP50, precision, recall, per-class AP, inference latency, training time, and peak GPU memory.
- Keep the generated YAML, manifests, histories, package version, and Kaggle notebook version with the report.